In [1]:
# Load model
from Deep_Library_BCI import MultiTaskModel
import tensorflow.keras as keras
model = MultiTaskModel(input_shape=(128,1), use_classifier=False, use_decoder=True)
model.build(input_shape=(128,1))
model.load_weights("../_Deep_Learning/Models_trained/2026-01-30-18_41_52_Autoencoder_1region_region_agnostic_First_Try/Split_1/model.weights.h5")

2026-01-31 12:52:19.897715: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-31 12:52:19.972756: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-31 12:52:21.544322: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1769860343.328595   46440 gpu_device.cc:2020] Created device /job:lo

8 128


In [15]:
import numpy as np
zeri = np.zeros(shape=(3,128,1))
model.encoder(zeri)

<tf.Tensor: shape=(3, 16), dtype=float32, numpy=
array([[-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542],
       [-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542],
       [-0.26309663, -0.06236269,  0.15675625, -0.15634431, -0.03892706,
         0.16875064, -0.16373911,  0.33522817, -0.18425772, -0.11028581,
         0.14779189, -0.03732343,  0.11379224,  0.01096453,  0.03833069,
        -0.15010542]], dtype=float32)>

This notebook is used to train a classifier that takes as features the ones extracted from Neural networks
and then train a cassifier on the top pf them 


In [ ]:
data_folder="../Data/"

import sys
import os

sys.path.append(os.path.abspath("Codes"))


from BCI_Library import read_subject, plot_brain, train_models, extract_features_more_bands, saveresults_pickle, select_rows
from BCI_Library import sort_rank, compute_welch_select_regions, extract_features_from_spectra_more_bands,compute_welch
from BCI_Library import plot_hist2d_selected_regions_folds, train_single_model, select_regions_MyCriteria, select_regions_Cohen_effect_size
from BCI_Library import global_selection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm



today = date.today()
sfreq = 250

seed = 224

saveresults = True
filepath = "../Results/BCI_Performances/--.pkl"
file_input = filepath
file_output = filepath
miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "Autoencoder extracted"
Region_Selection= "Cohen's d effect size selection" # "-"
Comments = "-"
PFR = False



n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

tutte_roi = 68
verbose = False

for DataType in ["EEG","MEG"]:
    for Classifier_name, meth in zip(["SVC", "RandomForest", "LDA"],[SVC, RandomForestClassifier, LDA]):

        Rows = pd.DataFrame({
            "ROIs": [],
            "NROIs":[],
            "DataType": [],
            "freqs_band": [],  
            "subject": [],
            "Classifier": [],
            "Performance": [],
            "Features": [],
            "Comments": [],
            "Region Selection": [],
            "Date": []
        })
       
        print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
        for subject_index in tqdm.tqdm(range(20)):

            data_2_MI = read_subject(data_folder, DataType, "MI", subject_index, verbose=False)
            data_2_Rest = read_subject(data_folder, DataType, "Baseline", subject_index, verbose=False)

            WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                                            kargs_welch={"fmin":miff[0], "fmax":maff[-1]})
            # WMI.shape (trials, ROIs, freqs)

            labels = [1 for __ in range(len(WMI))] + [0 for __ in range(len(WRe))] 

            Feat_extraction = extract_features_from_spectra_more_bands(WMI, WRe, [ _ for _ in range(tutte_roi)], FreqMI,
                    min_freq_frature=miff, max_freq_frature=maff, 
                    select_mean=True, select_max=True,
                    reshape_features=False)


            Feat_MI, Feat_Rest, labls_MI, labls_Rest, all_regions = Feat_extraction
            #print("\n\n"+"#"*30+"featureMI.shape",Feat_MI.shape)
            # Feat_MI.shape (Trials, selected_ROIs, features for Roi)

            Feat_All = np.concatenate((Feat_MI, Feat_Rest), axis=0)
            #print("\n\n"+"#"*30+"Feat_All.shape",Feat_All.shape)
            labels_All = np.array(labels)

            for num_best_ROIs in [3,5,10,20,40]:

                fold_accuracies = []

                for fold_idx, (train_idx, val_idx) in enumerate(kf.split([0 for _ in range(len(labels))], labels)):

                    # selecting region based on training data only
                    training_index_MI = train_idx[np.where(train_idx<len(WMI))]
                    training_index_Re = train_idx[np.where(train_idx>=len(WMI))]-len(WMI)

                    regions_selected, effect_size  = select_regions_Cohen_effect_size(wpsdMI=WMI[training_index_MI], 
                                                                    wpsdRest=WRe[training_index_MI], 
                                                                    important_regions=[], nbest_regions=num_best_ROIs)

                    
                    Feat_All_regions_selected = Feat_All[:,regions_selected,:].reshape(Feat_All.shape[0],-1)
                    #print("\n\n"+"#"*30+"Feat_All_regions_selected.shape",Feat_All_regions_selected.shape)
                    #####################################################################################################################

                    acc, cm, model = train_single_model(Feat_All_regions_selected, labels_All,
                                                        train_idx, val_idx, method=meth, seed=224)

                    fold_accuracies.append(acc)
                    if verbose:
                        print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                        print(cm)

                    #####################################################################################################################

                if verbose:
                    print("\n==================================\n","Mean accuracy:", np.mean(fold_accuracies))
                    print("Std accuracy:", np.std(fold_accuracies))

                new_row = {
                    "ROIs": [regions_selected],
                    "NROIs":[len(regions_selected)],
                    "DataType": [DataType],
                    "freqs_band": [freqs_band],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                    "Performance": [fold_accuracies],
                    "Features": [Features_name],
                    "Comments": [Comments],
                    "Region Selection": [Region_Selection],
                    "Date": [today],
                    "Personalized_Freq_Range": [PFR],
                }
                Rows=pd.concat((Rows,pd.DataFrame(new_row)))
        

        if saveresults:
            saveresults_pickle(Rows, inputfile=file_input, outputfile=file_output, backup=True)